# 16 — HTTP, JSON APIs, and Reliable Clients

HTTP clients request remote resources and translate them into application data. Learn methods, status handling, timeouts, JSON parsing, and testability. All runnable code uses local stubs—no network access or API tokens are required.

*Intermediate · 75–120 min*

## You will learn

- Construct requests and reason about GET and POST
- Decode JSON responses and validate schemas
- Set timeouts and handle HTTP/URL failures
- Mock remote calls and reason about pagination and retries

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. HTTP request model

GET normally retrieves data and should not change server state. POST generally submits data and may have side effects. Headers describe content and accepted formats; a timeout avoids waiting indefinitely. Never embed secret tokens in public notebooks.

In [ ]:
from urllib.request import Request

request = Request(
    "https://example.invalid/api/items?page=1",
    headers={"Accept": "application/json", "User-Agent": "PythonCourse/1.0"},
    method="GET",
)
print(request.get_method())
print(request.full_url)
print(request.get_header("Accept"))

**Think about it:** Why is a GET request not automatically harmless if the remote service is badly designed?

### 2. JSON decoding with a stub

A response body is bytes that must be decoded according to its encoding. Then JSON parsing yields Python values. The data may be a list or dictionary; always validate the expected schema. Here a BytesIO object acts as a fake response.

In [ ]:
import json
from io import BytesIO

body = BytesIO(b'{"items": [{"id": 1, "name": "Pen"}], "next": null}')
payload = json.loads(body.read().decode("utf-8"))
if not isinstance(payload, dict) or not isinstance(payload.get("items"), list):
    raise ValueError("unexpected response shape")
print(payload["items"])
print(payload["next"])

**Think about it:** What would happen if the response contained HTML instead of JSON?

### 3. Mock urlopen rather than using the network

By injecting a function that opens URLs, callers can use a mock during testing. This isolates HTTP behavior from data interpretation and prevents flaky tests that depend on external services.

In [ ]:
import json
from io import BytesIO
from urllib.request import Request

def get_json(url, *, opener, timeout=5):
    request = Request(url, headers={"Accept": "application/json"}, method="GET")
    with opener(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def fake_open(request, timeout):
    assert request.get_method() == "GET"
    assert timeout > 0
    return BytesIO(b'{"status": "ok", "count": 3}')

print(get_json("https://example.invalid/items", opener=fake_open))

**Think about it:** Which errors might a real opener raise that the fake never raises?

### 4. Pagination and controlled retry strategies

Many APIs return pages of results. An explicit next link or page token guides iteration. Clients should cap pages to avoid infinite loops. A retry policy may be useful for transient failures, but blindly retrying POST requests risks duplicate actions.

In [ ]:
PAGES = {
    1: {"items": ["A", "B"], "next": 2},
    2: {"items": ["C"], "next": None},
}
def collect_pages(fetch_page, first_page=1, max_pages=10):
    results, page, visited = [], first_page, set()
    while page is not None:
        if page in visited or len(visited) >= max_pages:
            raise RuntimeError("pagination loop or page limit reached")
        visited.add(page)
        response = fetch_page(page)
        results.extend(response["items"])
        page = response.get("next")
    return results

print(collect_pages(lambda page: PAGES[page]))

**Think about it:** Why is checking repeated page tokens useful?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/04_sql_http_cli_performance_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/16', {1: '58263c2fdeb10eef', 2: 'f8bc4d27b347c54c', 3: '0ef9e2502240c4a3', 4: '8d7ed47b8d604034'})

**Exercise 1.** Validate a JSON response: it must be a list of objects with a `sku`.

In [ ]:
import json

def validate_products(text):
    data = json.loads(text)
    if not isinstance(data, ___):
        return "not a list"
    bad = [i for i, item in enumerate(data) if ___]
    return "ok" if not bad else f"missing sku at {bad}"

check(1, [validate_products('[{"sku": "A"}]'), validate_products('{"sku": "A"}'), validate_products('[{"sku": 1}, {}]')])

**Exercise 2.** Inject the opener so the API wrapper can be tested without the network.

In [ ]:
import io, json

def fetch_json(url, opener):
    with ___ as response:
        return json.loads(response.read())

fake_opener = lambda url: io.BytesIO(json.dumps({"url": url, "ok": True}).encode())
check(2, fetch_json("https://api.example/items", ___))

**Exercise 3.** Follow `next` links, but stop after a maximum number of pages.

In [ ]:
pages = {1: {"items": [1, 2], "next": 2}, 2: {"items": [3], "next": 3}, 3: {"items": [4], "next": 1}}

def collect(fetch, max_pages):
    items, page, visited = [], 1, 0
    while page is not None and visited < ___:
        data = fetch(page)
        items.extend(data["items"])
        page = data["next"]
        visited += ___
    return items

check(3, collect(pages.get, max_pages=5))

**Exercise 4.** Respect `Retry-After` on status 429, but give up after a few tries.

In [ ]:
def wait_times(responses, max_attempts=3):
    waits = []
    for status, retry_after in responses[:___]:
        if status != 429:
            break
        waits.append(___)
    return waits

check(4, wait_times([(429, 1), (429, 2), (429, 4), (429, 8)]))

## Common mistakes

- Always set network timeouts in real clients.
- Do not make live network requests in deterministic teaching tests.
- Retries need limits and awareness of idempotency; never assume all POSTs can safely repeat.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why always set a timeout on HTTP calls?

   - **a)** To make requests faster
   - **b)** Servers require it
   - **c)** So a slow server can't hang your program forever

**2.** Which method should be safe to repeat with no side effects?

   - **a)** GET
   - **b)** Both equally
   - **c)** POST

**3.** Injecting the opener/transport into fetch functions lets you…

   - **a)** Skip JSON parsing
   - **b)** Use less memory
   - **c)** Test without the network

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/16', {1: '9697e9f328d9', 2: '641aae1308f5', 3: 'bb4f4680578a'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [17 — Command-Line Interfaces and Project Packaging](17_cli_and_packaging.ipynb)